# Cleaning and problem sizing

This notebook runs the cleaning and analysis steps of the pipeline in order and shows the intermediate results:

1. **Audit** the three raw files (`src/explore.py`)
2. **Clean** the daily file and add the derived daily columns (`src/clean.py`)
3. **Rebuild failure events** and check them against the official failure log (`src/events.py`)
4. **Size the problem**: failure rate, quick repeats, reactive spend, lost oil, concentration, failure mix (`src/sizing.py`)

The logic lives in `src/` so that `python run_all.py` gives exactly the same outputs as this notebook. Every number shown here is also written to a CSV in `outputs/`.

Raw data is read from `DATA_DIR` in `.env`. Nothing well-level is displayed, because the dataset may not be redistributed.

In [ ]:
import sys
from pathlib import Path

# The notebook runs from notebooks/, so add the repo root to the import path
# to make `src` importable.
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import pandas as pd

from src import clean, config, events, explore, sizing

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
for name, path in config.RAW_FILES.items():
    print(f"{name:<6} {path.name:<30} found={path.exists()}")

## 1. Audit the raw files

One row per (file, column) with type, null share and range. This is the evidence behind the cleaning rules. Shown below: columns more than half empty.

In [ ]:
audit = explore.main()
audit[audit.null_pct > 50].sort_values(["file", "null_pct"], ascending=[True, False])

## 2. Clean the daily file

Rules applied, in order:

- drop the exact duplicate rows
- null sensor placeholders (values at or above `PLACEHOLDER_MIN`)
- null negative volumes, implausible RPM and motor current, and pump efficiency outside 0 to 1.5
- flag `in_scope` (heavy-oil production rows) and `is_downhole` (downhole downtime, excluding disposal-well repairs)
- derive `is_producing`, `water_cut`, `oil_rate_m3d`, `rated_throughput_m3d` and `pump_utilisation`

No rows are filtered: every well is kept and later steps choose their population with `in_scope`. The log below accounts for every rule.

In [ ]:
log = clean.main()
log

### Derived columns on in-scope producing days

A sanity check that the new columns are in a believable range. Daily water cut is high, as expected for heavy oil. `pump_utilisation` divides oil only by the pump's rated liquid volume, so it is small: most of what the pump moves is water.

In [ ]:
con = config.connect()
sizing.register_views(con)

# Quantiles are computed in DuckDB so the 5M-row file never has to be loaded
# into pandas.
cols = ["water_cut", "oil_rate_m3d", "rated_throughput_m3d", "pump_utilisation"]
derived = con.sql(" union all ".join(
    f"""select '{c}' as column_name, count({c}) as n,
               quantile_cont({c}, 0.1) as p10, quantile_cont({c}, 0.5) as p50,
               quantile_cont({c}, 0.9) as p90, avg({c}) as mean
        from dc where in_scope and is_producing"""
    for c in cols
)).df()
derived

## 3. Rebuild failure events

Downhole down days on in-scope wells are merged into one event when they are `EVENT_GAP_DAYS` apart or closer (set in `.env`), because a single failure often shows up as several separate down days while the crew waits for a rig.

The events are then checked against the official failure log since 2025-07-01, both ways, and the whole build is repeated with a 7-day gap as a sensitivity check.

In [ ]:
val = events.main()
val

In [ ]:
ev = con.sql("select down_days, lost_oil_m3, matched_failure_log from ev").df()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Clip the long tail at 60 days so the bulk of the distribution is readable.
ev.down_days.clip(upper=60).plot.hist(bins=60, ax=axes[0], color="#4C78A8")
axes[0].set(title="Down days per rebuilt event (clipped at 60)", xlabel="down days", ylabel="events")

recall = val[(val.metric == "log_to_event_recall") & (val.gap_days == config.EVENT_GAP_DAYS)
             & val.group.isin(["all", "Rods", "Pump", "Tubing"])]
axes[1].bar(recall.group, recall.value, color="#4C78A8")
axes[1].set(title=f"Logged failures with a rebuilt event within {config.MATCH_WINDOW_DAYS} days",
            ylabel="share", ylim=(0, 1))
for x, v in zip(recall.group, recall.value):
    axes[1].text(x, v + 0.02, f"{v:.0%}", ha="center")
plt.tight_layout()

## 4. Size the problem

Each block prints its intermediate counts. The last block compares every headline value with the figure quoted in the brief and flags anything more than 1% off.

Notes for reading the output:

- **Failure rate** is events per producing well-year. The median run that ends in failure is shown separately and is biased short, so it is never quoted as the fleet rate.
- **Dollar figures** are `field_est_total` sums, est., CAD assumed. Zero-cost jobs count in sums but not in medians.
- **Pareto** is reported for two denominators. Quote one, and say which.

In [ ]:
sizing_df = sizing.main()

In [ ]:
sizing_df

In [ ]:
par = pd.read_csv(config.OUTPUTS / "pareto.csv")
mix = pd.read_csv(config.OUTPUTS / "failure_mix.csv")
s = sizing_df.set_index("metric").value

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Pareto curves: solid lines use all in-scope producers as the denominator,
# dashed lines only wells with at least one event.
for (denom, measure), g in par.groupby(["denominator", "measure"]):
    style = "-" if denom == "all_in_scope_producers" else "--"
    axes[0].plot(g.top_pct_wells, g.cumulative_share, style, marker="o", label=f"{measure}, {denom}")
axes[0].plot([0, 100], [0, 1], color="grey", lw=0.8)
axes[0].set(title="Concentration by well", xlabel="top % of wells", ylabel="cumulative share")
axes[0].legend(fontsize=7)

# Reactive versus planned spend, stacked per job category.
cats = list(config.REPAIR_JOB_CATEGORIES)
reactive = [s[f"reactive_spend_{c}"] / 1e6 for c in cats]
planned = [(s[f"spend_{c}"] - s[f"reactive_spend_{c}"]) / 1e6 for c in cats]
axes[1].bar(cats, reactive, label="reactive", color="#E45756")
axes[1].bar(cats, planned, bottom=reactive, label="planned", color="#4C78A8")
axes[1].set(title="Repair spend, $M (est., CAD assumed)", ylabel="$M")
axes[1].legend()

# Failure mix by year from the official log; 2021 and 2022 have one record each.
pivot = mix.pivot_table(index="year", columns="failure_type", values="failures", aggfunc="sum", fill_value=0)
pivot.plot.bar(stacked=True, ax=axes[2])
axes[2].set(title="Official failure log by year (descriptive only)", xlabel="", ylabel="records")
axes[2].legend(fontsize=7)
plt.tight_layout()

## Headline numbers for the slides

Only numbers printed by this cell (or present in `outputs/sizing.csv`) go on a slide.

In [ ]:
headline = [
    ("Failures per producing well-year", f"{s['failures_per_well_year']:.2f}"),
    ("Producing days per failure", f"{s['producing_days_per_event']:.1f}"),
    ("Median failed run (failed runs only, biased short)",
     f"{s['median_failed_run_producing_days']:.0f} producing / {s['median_failed_run_calendar_days']:.0f} calendar days"),
    ("Next failure within 30 calendar days", f"{s['quick_repeat_share_calendar']:.1%}"),
    ("Reactive share of repair jobs", f"{s['reactive_job_share']:.1%}"),
    ("Reactive share of repair spend (est., CAD assumed)",
     f"{s['reactive_spend_share']:.1%} of ${s['repair_spend'] / 1e6:,.1f}M"),
    ("Lost oil on downhole events",
     f"{s['downhole_lost_oil_m3']:,.0f} m3 ({s['downhole_lost_oil_bbl'] / 1e6:.2f}M bbl), "
     f"{s['downhole_share_of_lost_oil']:.0%} of all lost oil"),
    ("Top 10% of all in-scope producers",
     f"{s['pareto_all_top10_failures']:.1%} of failures, {s['pareto_all_top10_lost_oil']:.1%} of lost oil"),
    ("Top 10% of wells with at least one event",
     f"{s['pareto_event_top10_failures']:.1%} of failures, {s['pareto_event_top10_lost_oil']:.1%} of lost oil"),
]
pd.DataFrame(headline, columns=["metric", "value"])

## Conclusion

### Identified problem

**Heavy-oil PCP wells fail downhole often, fail again soon after repair, and are almost always fixed after the fact. Most of the cost and lost oil sits in a minority of wells.**

| Finding | Value | Source |
|---|---|---|
| Failure rate | 2.15 failures per producing well-year (one every 170 producing days) | `sizing.csv` |
| Quick repeats | 28.8% of restarts fail again within 30 calendar days | `sizing.csv` |
| Reactive work | 80.4% of rod and workover jobs, and 77.2% of their $212.7M spend (est., CAD assumed), follow a downhole failure | `sizing.csv` |
| Lost oil | 208,840 m3 (1.31M bbl) on downhole events, 37% of all lost oil | `sizing.csv` |
| Concentration | The top 10% of in-scope producers account for 40.8% of failures and 51.3% of downhole lost oil | `pareto.csv` |

### How we got there

1. **Cleaned** 5.08M well-days: removed 29 duplicate rows, nulled placeholders and impossible values, and kept every well. Each rule is logged in `cleaning_log.csv`.
2. **Scoped** the analysis to heavy-oil production rows (1,863 wells, 1.79M producing well-days), because these are the wells that run PCPs.
3. **Rebuilt failures** from downtime codes, merging downhole down days 4 or fewer days apart into one event: 10,548 events on 1,191 wells. The official failure log only starts in mid-2025, so it was used to check the rebuild, not to define it. 77% of logged failures match a rebuilt event within 7 days, and the result holds with a 7-day merge gap (10,082 events, 73% match).
4. **Sized** the problem using producing days as exposure, so shut-in time does not make wells look healthier. A job counts as reactive if the well was already down for a downhole reason in the 14 days before it started.

### From problem to research question

The problem is repetition. Nearly 3 in 10 repairs are followed by another failure within a month, and four out of five jobs are reactive. That means the lever is the **time a well runs between downhole failures**, and how the repair and the operating conditions change it. The rebuilt events give that run time for every well across five years, which the failure log alone could not.

> **Research question.** Which operating conditions and repair choices shorten or extend the time a heavy-oil PCP well runs between downhole failures, and is the extra run time from a more thorough repair worth its cost?

Next steps: build one row per run between events (`runs.py`) with features measured before each run starts, model run life with survival analysis, and compare repair types against matched controls to price the extra run time.